# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliLabib2006/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I chose **Lane 2: Refresh / Content Opportunity Scoring**. The goal of this lane is to find which content pages should be reviewed first for refreshing, expanding, improving, or monitoring. I chose it because the dataset contains useful page-level information such as impressions, clicks, sessions, CTR, average position, content age, engagement, and trend direction. These signals may help create a ranked list so that a content or SEO team can focus on the most important pages first.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane = "Refresh / Content Opportunity Scoring"

print("My provisional lane is:")
print(lane)

My provisional lane is:
Refresh / Content Opportunity Scoring


## 2. The question: decision, action, cost of a wrong call

### Research question

Which content pages should be reviewed first because they show signs of decline or a useful improvement opportunity?

### Unit of analysis

The unit of analysis is **one content page**. Each row in the dataset represents one anonymized content page.

### Output

The output will be a ranked list of pages from highest priority to lowest priority. Each page can also have a reason explaining why it was selected, such as low CTR, declining traffic, old content, or weak engagement.

### Decision and action

The ranked list will help a content or SEO team decide which pages to review first.

After reviewing a page, the team may choose to:

- update old information;
- expand thin content;
- improve the title or description;
- improve engagement;
- monitor the page;
- merge or remove weak content.

The system only recommends which pages should be reviewed first. A human makes the final decision.

### Cost of a wrong recommendation

A false positive happens when the system recommends a page that does not need improvement. This wastes the reviewer’s time and could lead to an unnecessary change.

A false negative happens when the system misses a page that really needs attention. The team may miss an important opportunity, and the page may continue losing traffic.

### Why data or machine learning can help

A website may contain thousands of pages, so a person cannot review every page manually. Data analysis can combine signals such as impressions, clicks, CTR, average position, sessions, page age, engagement, and trend direction.

Machine learning may help rank pages better than using only one simple rule. However, the model must be compared with a clear rule-based baseline and checked by a human reviewer.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [15]:
import os

print("Current folder:", os.getcwd())

!find /content -name "content_refresh_anonymized.csv"

Current folder: /content/flyrank-internship-ml-01
/content/flyrank-internship-ml-01/data/raw/content_refresh_anonymized.csv


In [16]:
!git clone https://github.com/AliLabib2006/flyrank-internship-ml-01

Cloning into 'flyrank-internship-ml-01'...
remote: Enumerating objects: 122, done.
remote: Counting objects: 100% (122/122), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 122 (delta 36), reused 95 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (122/122), 1.85 MiB | 15.00 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [17]:
!ls

02_your_first_readable_model.ipynb  flyrank-internship-ml-01  requirements.txt
AGENTS.md			    GUIDE.md		      scripts
CLAUDE.md			    LICENSE		      SETUP.md
data				    notebooks		      skills
DATA_USE.md			    outputs		      submission
docs				    README.md		      work


In [18]:
%cd flyrank-internship-ml-01

/content/flyrank-internship-ml-01/flyrank-internship-ml-01


In [19]:
!ls data/raw

content_refresh_anonymized.csv


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

I will use the starter dataset to check whether there are enough meaningful
content opportunities for this lane.

I focus on pages that have at least one impression, are at least 90 days old,
and are unique by content ID. The code below shows:

1. the number of eligible pages;
2. the number and percentage of declining pages;
3. the number and percentage of declining pages with at least 100 impressions.

Pages that are both declining and receiving impressions may be more useful to
review than pages with little or no visibility.

In [20]:
from pathlib import Path
import pandas as pd


# Find the starter dataset inside the cloned repository.
matches = list(Path("/content").rglob("content_refresh_anonymized.csv"))

if not matches:
    matches = list(Path.cwd().rglob("content_refresh_anonymized.csv"))

if not matches:
    raise FileNotFoundError(
        "content_refresh_anonymized.csv was not found. "
        "Make sure the repository is cloned in Colab."
    )

data_path = matches[0]

# Load the dataset.
df = pd.read_csv(data_path)

# Check that the required columns exist.
required_columns = {
    "content_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction",
}

missing_columns = required_columns - set(df.columns)

if missing_columns:
    raise KeyError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

# Apply the starter project filtering rules.
analysis_df = (
    df.loc[
        (df["impressions_90d"] > 0)
        & (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

# Create simple opportunity indicators.
analysis_df["is_declining"] = (
    analysis_df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
)

analysis_df["declining_with_demand"] = (
    analysis_df["is_declining"]
    & (analysis_df["impressions_90d"] >= 100)
)

# Calculate three real numbers.
eligible_pages = len(analysis_df)
declining_pages = int(analysis_df["is_declining"].sum())
declining_with_demand = int(
    analysis_df["declining_with_demand"].sum()
)

declining_percentage = (
    declining_pages / eligible_pages * 100
    if eligible_pages > 0
    else 0
)

demand_percentage = (
    declining_with_demand / eligible_pages * 100
    if eligible_pages > 0
    else 0
)

summary = pd.DataFrame({
    "Measure": [
        "Eligible unique pages",
        "Declining pages",
        "Declining pages with at least 100 impressions",
    ],
    "Count": [
        eligible_pages,
        declining_pages,
        declining_with_demand,
    ],
    "Percentage of eligible pages": [
        100.0,
        declining_percentage,
        demand_percentage,
    ],
})

display(summary)

print(f"Dataset loaded from: {data_path}")
print(f"Eligible unique pages: {eligible_pages:,}")
print(
    f"Declining pages: {declining_pages:,} "
    f"({declining_percentage:.2f}%)"
)
print(
    f"Declining pages with at least 100 impressions: "
    f"{declining_with_demand:,} ({demand_percentage:.2f}%)"
)

,Measure,Count,Percentage of eligible pages
0,Eligible unique pages,30000,100.000000
1,Declining pages,16262,54.206667
2,Declining pages with at least 100 impressions,13152,43.840000


Dataset loaded from: /content/flyrank-internship-ml-01/data/raw/content_refresh_anonymized.csv
Eligible unique pages: 30,000
Declining pages: 16,262 (54.21%)
Declining pages with at least 100 impressions: 13,152 (43.84%)


After applying the starter filtering rules, the dataset contains
**[30000] eligible unique pages**.

Among them, **[16262] pages ([54.206667]%)** are marked as declining.

There are also **[13152] declining pages ([43.840000]%)** with at least
100 impressions.

These observed numbers suggest that there are enough possible review
candidates to make ranking useful. A content team may not have enough time to
review every page, so a ranked review queue could help it focus on stronger
opportunities first.

## 4. Careful words: what I can and can't claim

My work will be able to describe observed patterns in the anonymized dataset
and provide directional, decision-support recommendations.

It may help identify pages that appear more suitable for human review based on
measured signals such as impressions, clicks, CTR, position, content age,
sessions, engagement, and trend direction.

The result will be a ranking of pages to review first. It will not be an
automatic decision to edit, remove, or refresh a page.

This work cannot prove that refreshing a page will cause its traffic to
increase. It also cannot prove Google ranking factors, predict changes to
Google's algorithm, or guarantee future performance.

The current decline label is based on the available observation window, so it
is a simple proxy rather than a perfect future outcome. Stronger future work
would use an earlier feature window and a separate later target window.

All conclusions will use careful language such as “observed,” “measured,”
“suggests,” “directional,” and “decision-support.” Human review is still
required before any action is taken.

In [21]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Check that this notebook uses only safe fields for the Week 1 analysis.

used_columns = {
    "content_id",
    "impressions_90d",
    "content_age_days",
    "trend_direction",
}

private_terms = {
    "client_name",
    "domain",
    "raw_url",
    "raw_query",
    "private_query",
}

unsafe_used_columns = used_columns.intersection(private_terms)

assert not unsafe_used_columns, (
    f"Unsafe fields detected: {sorted(unsafe_used_columns)}"
)

print("Careful-claims and privacy check passed.")
print("The analysis uses anonymized IDs and aggregated measurements only.")


Careful-claims and privacy check passed.
The analysis uses anonymized IDs and aggregated measurements only.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.